In this notebook, we will document and map every found issue with GDPR and AI act. 

# Notebook 03 — Data Governance & Compliance Audit
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook performs a structured compliance audit of the 500 cleaned credit application records produced by Notebook 01. Every data quality issue identified in the cleaning pipeline is mapped to its corresponding obligations under the **General Data Protection Regulation (GDPR)** and the **EU AI Act**, the two primary regulatory frameworks governing the collection, processing, and automated decision-making of personal data in the European Union. For each issue, this notebook documents the legal basis of the violation, assesses its severity, and provides concrete policy recommendations and remediation actions for NovaCred's Data Governance Officer.

### Regulatory Framework

**GDPR (Regulation 2016/679)** governs how personal data of EU individuals is collected, stored, processed, and protected. In the context of credit applications, virtually every field — name, email, SSN, date of birth, income — constitutes personal data and is subject to GDPR obligations including data minimisation, accuracy, storage limitation, and integrity.

**EU AI Act (Regulation 2024/1689)** classifies credit scoring as a **high-risk AI application** under Annex III. This means NovaCred is legally required to ensure the quality of data used to train and operate its credit scoring model, maintain human oversight over automated decisions, and keep detailed documentation of data quality issues and how they were resolved.

### Scope of This Audit

This audit covers the 14 data quality issues catalogued in Notebook 01, organised across five compliance dimensions:

| Dimension | Issues Covered |
|-----------|---------------|
| Uniqueness | Duplicate records, conflicting SSNs |
| Consistency | Gender coding, date formats |
| Validity | Income type, negative credit history, DTI ratio |
| Completeness | Missing income, DOB, email, SSN, timestamp |
| Accuracy | Age plausibility, timestamp plausibility, cross-field plausibility |

> **Note** — This audit uses the cleaned dataset `cleaned_credit_applications.csv` produced by Notebook 01 as its primary input. All flags and audit trail columns created during cleaning are used directly as evidence in this compliance analysis.

In [15]:
import pandas as pd
import regex as re

df_raw = pd.read_csv('../data/raw_credit_applications.csv')
df_clean = pd.read_csv('../data/cleaned_credit_applications.csv')

## Section 1 — Uniqueness

### #1 Duplicate Records & #2 Conflicting SSNs

In [9]:
# ─── #1: Duplicate _id records ───────────────────────────────────────────────
print('── #1 Duplicate Records ──')
print()
print('Raw dataset:')
dup_raw = df_raw[df_raw['_id'].duplicated(keep=False)][['_id', 'full_name', 'email', 'ssn']]
print(dup_raw.to_string(index=False))
print()
print('After cleaning:')
dup_clean = df_clean[df_clean['_id'].duplicated(keep=False)][['_id', 'full_name', 'email', 'ssn']]
if len(dup_clean) == 0:
    print('  No duplicate _id records found ✓')
print()

print('Affected rows in df_raw:')
print(dup_ids.to_string(index=False))
print()

print(f'Records in df_raw  : {len(df_raw)}')
print(f'Records in df_clean: {len(df_clean)}')
print(f'Removed            : {len(df_raw) - len(df_clean)} duplicate rows')


# ─── #2: Conflicting SSNs ────────────────────────────────────────────────────
print()
print('── #2 Conflicting SSNs ──')
print()

# Find SSNs that appear more than once
dup_ssns = df_raw['ssn'].value_counts()
dup_ssns = dup_ssns[dup_ssns > 1].index.tolist()

print('BEFORE (df_raw):')
print(df_raw[df_raw['ssn'].isin(dup_ssns)][['_id', 'full_name', 'ssn']].to_string(index=False))
print()
print('AFTER (df_clean):')
print(df_clean[df_clean['ssn'].isin(dup_ssns)][['_id', 'full_name', 'ssn']].to_string(index=False))
print()
print('  Classification:')
for ssn in dup_ssns:
    rows  = df_raw[df_raw['ssn'] == ssn]
    ids   = rows['_id'].tolist()
    names = rows['full_name'].tolist()
    if len(set(ids)) == 1:
        print(f'  {ssn} → Case A: exact duplicate row, handled by dedup')
    elif len(set(names)) == 1:
        print(f'  {ssn} → Case B: same person, two applications — flag for review')
    else:
        print(f'  {ssn} → Case C: different people, same SSN — escalate to legal')


── #1 Duplicate Records ──

Raw dataset:
    _id        full_name                       email         ssn
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com 427-90-1892
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com         NaN

After cleaning:
  No duplicate _id records found ✓

Affected rows in df_raw:
    _id        full_name                       email         ssn
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com 427-90-1892
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com         NaN

Records in df_raw  : 502
Records in df_clean: 500
Removed            : 2 duplicate rows

── #2 Conflicting SSNs ──

BEFORE (df_raw):
    _id      full_name         ssn
app_042   Joseph Lopez 652-70-5530
app_101   Sandra Smith 937-

### Compliance Analysis — Uniqueness

#### #1 — Duplicate Records

**GDPR — Article 5(1)(d) — Accuracy**
Personal data must be accurate and kept up to date. Duplicate records mean the same individual appears twice in the system, which can lead to incorrect decisions being made on stale or redundant data.

**GDPR — Article 5(1)(c) — Data Minimisation**
Data must be adequate, relevant, and limited to what is necessary. Retaining duplicate records violates this principle — NovaCred should not hold more records about an individual than necessary.

**Recommended Actions:**
- **Data Engineering Team**: Implement a unique constraint on `_id` at the point of data ingestion — reject any application whose ID already exists in the system.
- **Data Engineering Team**: Schedule periodic deduplication audits on the live database to catch any duplicates that slip through.

---

#### #2 — Conflicting SSNs

**Case A — Exact duplicate row (same `_id`, same name)**
This is purely a data pipeline issue, not an SSN conflict. The SSN itself is fine.
- **GDPR — Article 5(1)(c) — Data Minimisation**: redundant records must not be retained.
- **Action**: already resolved by deduplication in Notebook 01. No further action needed on the SSN.

**Case B — Same person, two applications (different `_id`, same name)**
A single person submitted two separate applications. This is not necessarily fraudulent but raises questions about whether both applications were processed independently.
- **GDPR — Article 5(1)(b) — Purpose Limitation**: data collected for one credit application should not be silently reused for a second without the applicant's knowledge.
- **Action**: Both records must be reviewed manually. If the second application was intentional, it should be processed normally. If it was a system error, one record should be removed and the applicant notified.

**Case C — Different people, same SSN (different `_id`, different name)**
Two different individuals claim the same Social Security Number. This is the most serious finding — it is either a data entry error or an indicator of identity fraud.
- **GDPR — Article 5(1)(d) — Accuracy**: at least one of these records contains incorrect personal data.
- **GDPR — Article 33 — Data Breach Notification**: if identity fraud is confirmed, NovaCred is legally obligated to notify the relevant supervisory authority within 72 hours.
- **EU AI Act — Article 10 — Data Governance**: corrupted identity data must not be used in any credit scoring model.
- **Action**: Both records must be immediately excluded from model training and any automated credit decision. The case must be escalated to NovaCred's legal team and, if fraud is confirmed, reported to the relevant authority.

## Section 2 - Consistency

### #3 Gender Coding & #4 Date Formats

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# Consistency: Gender Coding & Date Formats
# ═══════════════════════════════════════════════════════════════════════════

# ─── #3: Gender Coding ───────────────────────────────────────────────────────
print('── #3 Gender Coding ──')
print()
print('BEFORE (df_raw) — inconsistent values:')
print(df_raw['gender_raw'].fillna('NULL').value_counts().to_string())
print()
print('AFTER (df_clean) — standardised values:')
print(df_clean['gender'].value_counts(dropna=False).to_string())

# ─── #4: Date Formats ────────────────────────────────────────────────────────
print()
print('── #4 Date Formats ──')
print()

# Detect format of each raw date string on the fly
def detect_format(d):
    if not d:
        return 'MISSING'
    if re.match(r'^\d{4}-\d{2}-\d{2}$', str(d)): return 'YYYY-MM-DD'
    if re.match(r'^\d{4}/\d{2}/\d{2}$', str(d)): return 'YYYY/MM/DD'
    if re.match(r'^\d{2}/\d{2}/\d{4}$', str(d)): return 'DD/MM/YYYY or MM/DD/YYYY'
    return 'UNKNOWN'

df_raw['dob_format_temp'] = df_raw['date_of_birth_raw'].apply(detect_format)

print('BEFORE (df_raw) — multiple formats detected:')
print(df_raw['dob_format_temp'].value_counts().to_string())
print()
print('Examples of each format:')
for fmt in df_raw['dob_format_temp'].unique():
    example = df_raw[df_raw['dob_format_temp'] == fmt]['date_of_birth_raw'].iloc[0]
    print(f'  {fmt:<30} → e.g. "{example}"')

print()
print('AFTER (df_clean) — all dates in ISO 8601:')
all_uniform = df_clean['date_of_birth'].dropna().apply(lambda x: str(x)[:10]).str.match(r'^\d{4}-\d{2}-\d{2}$').all()
print(df_clean['date_of_birth'].dropna().apply(lambda x: str(x)[:10]).head(5).to_string())
print()
print(f'  All parsed dates follow YYYY-MM-DD format: {all_uniform} ✓')

# clean up temp column
df_raw.drop(columns=['dob_format_temp'], inplace=True)

── #3 Gender Coding ──

BEFORE (df_raw) — inconsistent values:
gender_raw
Male      195
Female    193
F          58
M          53
NULL        3

AFTER (df_clean) — standardised values:
gender
Female    250
Male      247
NaN         3

── #4 Date Formats ──

BEFORE (df_raw) — multiple formats detected:
dob_format_temp
YYYY-MM-DD                  340
DD/MM/YYYY or MM/DD/YYYY    101
YYYY/MM/DD                   56
UNKNOWN                       5

Examples of each format:
  YYYY-MM-DD                     → e.g. "2001-03-09"
  DD/MM/YYYY or MM/DD/YYYY       → e.g. "14/02/1982"
  YYYY/MM/DD                     → e.g. "1990/07/26"
  UNKNOWN                        → e.g. "nan"

AFTER (df_clean) — all dates in ISO 8601:
0    2001-03-09
1    1992-03-31
2    1989-10-24
3    1983-04-25
4    1999-05-21

  All parsed dates follow YYYY-MM-DD format: True ✓


### Compliance Analysis — Consistency

#### #3 — Inconsistent Gender Coding

**GDPR — Article 5(1)(d) — Accuracy**
Gender is personal data under GDPR. Storing the same value in four different formats (`Male`, `M`, `Female`, `F`) means the data is not standardised and cannot be reliably processed. Any system that reads this field — including the credit scoring model — may treat `M` and `Male` as different categories, introducing silent errors into automated decisions.

**EU AI Act — Article 10 — Data Quality**
High-risk AI systems must use training data that is consistent and free from encoding errors. Inconsistent gender coding directly violates this requirement — a model trained on this data would receive contradictory signals for what is effectively the same value.

**Consequences if unresolved:**
- The credit scoring model may produce different outcomes for applicants with `M` vs `Male`, constituting an unintended bias.
- Any statistical analysis of gender distribution across approvals/rejections would produce incorrect results, undermining fairness reporting obligations.

**Recommended Actions:**
- **Data Engineering Team**: Implement input validation at ingestion — the `gender` field should only accept `Male` or `Female` (or a defined set of values). Any other value should be rejected at entry.
- **Data Engineering Team**: Add a standardisation step to the ingestion pipeline that maps abbreviations to full words automatically before the record enters the database.

---

#### #4 — Mixed Date Formats

**GDPR — Article 5(1)(d) — Accuracy**
Date of birth is sensitive personal data. When stored in four different formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM/DD/YYYY`, `YYYY/MM/DD`), there is a genuine risk of misinterpretation — for example, `03/04/1990` could be read as March 4th or April 3rd depending on the assumed format. This means the system may be operating on an incorrect date of birth for a subset of applicants.

**EU AI Act — Article 10 — Data Quality**
Inconsistent date formats are a data quality defect that directly affects the accuracy of derived features such as age. If the credit scoring model uses age as an input feature, misinterpreted dates of birth will silently corrupt the model's inputs, leading to potentially incorrect credit decisions.

**Consequences if unresolved:**
- Applicants could be incorrectly assessed as older or younger than they are, affecting credit eligibility.
- The 5 records classified as `UNKNOWN` format have dates that could not be reliably interpreted — these applicants' birth dates are effectively unknown to the system despite appearing to have data.

**Recommended Actions:**
- **Data Engineering Team**: Enforce ISO 8601 (`YYYY-MM-DD`) as the only accepted date format at the point of data ingestion. Any other format should be rejected with a clear error message returned to the applicant.
- **Data Engineering Team**: The 5 records with `UNKNOWN` format must be manually reviewed. Their date of birth cannot be reliably used until the correct value is confirmed with the applicant.

# Section 3 - Validity

## #5 Income type, #6 Negative credit history, #7 DTI ratio

In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# Validity: Income Type, Negative Credit History, DTI Ratio
# ═══════════════════════════════════════════════════════════════════════════

# ─── #5: Income Stored as Text ───────────────────────────────────────────────
print('── #5 Income Stored as Text ──')
print()
print('BEFORE (df_raw) — income stored as text:')
type_breakdown = df_raw['annual_income_raw'].apply(type).value_counts()
print(type_breakdown.to_string())
print()
print('AFTER (df_clean) — all values numeric:')
print(f'  dtype in df_clean: {df_clean["annual_income"].dtype}')
print(f'  Non-numeric values remaining: {df_clean["annual_income"].apply(lambda x: isinstance(x, str)).sum()} ✓')

# ─── #6: Negative Credit History ─────────────────────────────────────────────
print()
print('── #6 Negative Credit History ──')
print()
print('BEFORE (df_raw) — negative values:')
neg_credit = df_raw[df_raw['credit_history_months'] < 0][['_id', 'credit_history_months']]
print(neg_credit.to_string(index=False))
print()
print('AFTER (df_clean) — absolute values applied:')
print(df_clean[df_clean['credit_history_months_flag'] == True][['_id', 'credit_history_months', 'credit_history_months_flag']].to_string(index=False))
print()
print(f'  Minimum credit_history_months in df_clean: {df_clean["credit_history_months"].min()} ✓')

# ─── #7: Impossible DTI Ratio ────────────────────────────────────────────────
print()
print('── #7 Impossible DTI Ratio ──')
print()
print('BEFORE (df_raw) — DTI outside valid range [0, 1]:')
bad_dti = df_raw[df_raw['debt_to_income'] > 1.0][['_id', 'debt_to_income']]
print(bad_dti.to_string(index=False))
print()
print('AFTER (df_clean) — corrected:')
print(df_clean[df_clean['dti_flag'] == True][['_id', 'debt_to_income', 'dti_flag']].to_string(index=False))
print()
print(f'  Maximum DTI in df_clean: {df_clean["debt_to_income"].max()} ✓')

── #5 Income Stored as Text ──

BEFORE (df_raw) — income stored as text:
annual_income_raw
<class 'float'>    502

AFTER (df_clean) — all values numeric:
  dtype in df_clean: float64
  Non-numeric values remaining: 0 ✓

── #6 Negative Credit History ──

BEFORE (df_raw) — negative values:
    _id  credit_history_months
app_043                    -10
app_156                     -3

AFTER (df_clean) — absolute values applied:
    _id  credit_history_months  credit_history_months_flag
app_043                     10                        True
app_156                      3                        True

  Minimum credit_history_months in df_clean: 0 ✓

── #7 Impossible DTI Ratio ──

BEFORE (df_raw) — DTI outside valid range [0, 1]:
    _id  debt_to_income
app_402            1.85

AFTER (df_clean) — corrected:
    _id  debt_to_income  dti_flag
app_402           0.185      True

  Maximum DTI in df_clean: 0.45 ✓


### Compliance Analysis — Validity

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
*"Personal data shall be accurate and, where necessary, kept up to date; every reasonable step must be taken to ensure that personal data that are inaccurate, having regard to the purposes for which they are processed, are erased or rectified without delay."*

All three validity issues identified in this section — income stored as text, negative credit history, and an impossible DTI ratio — constitute accuracy violations under this article. In each case, the field contains a value that is either technically incorrect (wrong data type) or factually impossible (negative months, ratio above 1.0), meaning NovaCred is processing and potentially making credit decisions based on inaccurate personal data.

**EU AI Act — Article 10(3) — Data and Data Governance**
*"Training, validation and testing data sets shall be relevant, sufficiently representative, and to the best extent possible, free of errors and complete in view of the intended purpose."*

Credit scoring is classified as a high-risk AI application under Annex III of the EU AI Act. All three issues directly compromise the quality of data used to train and operate the model — a type error in income, an impossible credit history value, and an out-of-range DTI ratio would each corrupt the model's inputs and potentially lead to incorrect or discriminatory credit decisions.

**EU AI Act — Article 14(1) — Human Oversight**
*"High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine interface tools, that they can be effectively overseen by natural persons during the period in which the AI system is in use."*

Any automated correction applied to these fields — such as the absolute value fix for negative credit history or the division by 10 for the DTI ratio — must be logged, documented, and formally approved by the Governance Officer before the corrected records are used in any model training or credit decision.

---

#### Issue Breakdown

**#5 — Income Stored as Text**
Storing `"55000"` as a string instead of a number means the value cannot be reliably used in numerical computation. A system that fails to convert it correctly would either crash or silently treat the income as null, leading to an incorrect credit assessment. Applicants with text-encoded income risk being assessed as having no income, resulting in a wrongful rejection.

**#6 — Negative Credit History Months**
A value of `-10 months` is factually impossible — credit history cannot be negative. This is either a data entry error or a system bug. If fed directly into the credit scoring model, it would be interpreted as an extremely poor credit profile, leading to an unjust rejection of the affected applicant.

**#7 — Impossible Debt-to-Income Ratio**
A DTI of `1.85` implies the applicant's debt is 185% of their income, which is mathematically impossible as a ratio. The most likely explanation is a decimal point error (`1.85` instead of `0.185`). If uncorrected, the model would treat this applicant as catastrophically over-indebted, virtually guaranteeing an incorrect rejection.

---

#### Consequences if Unresolved
- Applicants affected by these errors could receive incorrect loan rejections based on corrupted data, constituting unfair automated decisions under **GDPR Article 22**.
- The credit scoring model trained on this data would learn distorted relationships between features and creditworthiness, undermining the reliability of all future decisions.
- NovaCred would be in breach of its EU AI Act obligations regarding data quality for high-risk AI systems.

---

#### Recommended Actions
- **Data Engineering Team**: Enforce strict type validation on `annual_income` at ingestion — reject any value that cannot be cleanly cast to a number.
- **Data Engineering Team**: Implement range validation at ingestion — `credit_history_months` must be a non-negative integer and `debt_to_income` must be between `0.0` and `1.0`. Any value outside these ranges should be rejected and flagged for manual review.
- **Governance Officer**: formally approve and document the corrections applied in Notebook 01 (`abs()` for credit history, `÷10` for DTI) before the affected records are used in any model training or credit decision.